In [2]:
!pip install pandas

   ---------------------------------------- 0.0/11.3 MB ? eta -:--:--
   ---------------------------------------- 0.0/11.3 MB ? eta -:--:--
   --- ------------------------------------ 1.0/11.3 MB 5.0 MB/s eta 0:00:03
   --------------- ------------------------ 4.5/11.3 MB 11.7 MB/s eta 0:00:01
   -------------------------------------- - 11.0/11.3 MB 18.6 MB/s eta 0:00:01
   ---------------------------------------- 11.3/11.3 MB 18.2 MB/s  0:00:00

   ---------------------------------------- 0/3 [pytz]
   ------------- -------------------------- 1/3 [tzdata]
   ------------- -------------------------- 1/3 [tzdata]
   -------------------------- ------------- 2/3 [pandas]
   -------------------------- ------------- 2/3 [pandas]
   -------------------------- ------------- 2/3 [pandas]
   -------------------------- ------------- 2/3 [pandas]
   -------------------------- ------------- 2/3 [pandas]
   -------------------------- ------------- 2/3 [pandas]
   -------------------------- --------

In [3]:
# =====================================================================================
# پروژه: PTB-XL Metadata Audit  —  ممیزیِ فقط-خواندنیِ یکپارچگی و متادیتا
# نسخه کد: v0.1.1   (اصلاحِ v0.1)
# تاریخ: 2026-09-09
# هدف: ثبتِ provenance، شکلِ داده، متادیتای ضبط، دموگرافیک، سلامتِ split،
#      اشتقاقِ برچسبِ ابرکلاس، و جدول‌های تقاطعیِ device/site/nurse در برابر fold و برچسب.
# متغیر مستقل: هیچ — این نسخه هیچ مدلی آموزش نمی‌دهد و هیچ عددِ عملکردی تولید نمی‌کند.
#
# تغییرات نسبت به v0.1:
#   ۱) راستی‌آزماییِ هشِ محاسبه‌شده در برابر SHA256SUMS (فقط دو CSV؛ فایل‌های سیگنالی هش نمی‌شوند)
#   ۲) جدول‌های device/site × ابرکلاس فقط روی folds 1–9؛ روی فولدِ ۱۰ فقط شیوعِ حاشیه‌ای
#      (خطِ پایه‌ی شانسِ AUPRC) محاسبه می‌شود و هیچ جدولِ مشترکِ متادیتا×برچسبی روی آن ساخته نمی‌شود
#   ۳) بخشِ ۷.۵: دسته‌های دیده‌نشده‌ی device/site/nurse بین foldها
#   ۴) nurse × strat_fold با Cramér's V خام و تصحیح‌شده
#   ۵) نسخه‌ی خام (بدونِ گردکردن) هر جدولِ نسبتی با پسوندِ _raw
#   ۶) افزودنِ n_changed و fold_tab به manifest
#   ۷) برچسب‌های صفرپرشده برای device/site/nurse (مثلاً site_007) برای مرتب‌سازیِ درست
#   ۸) بنرِ پرسروصدای نشت بلافاصله بعد از بخش ۵ (پرتابِ AssertionError همچنان در انتها)
#
# ورودی‌های مجاز (read-only): ptbxl_database.csv | scp_statements.csv | RECORDS |
#                              SHA256SUMS | ptbxl_v102_changelog.txt | ptbxl_v103_changelog.txt
# هیچ فایلِ سیگنالی از records100/ یا records500/ باز یا هش نمی‌شود. wfdb ایمپورت نمی‌شود.
#
# خروجی‌ها (همه در OUT_DIR):
#    1) v0.1.1_column_inventory.csv
#    2) v0.1.1_labels.csv
#    3) v0.1.1_crosstab_device_fold.csv                      + _rowshare_raw
#    4) v0.1.1_crosstab_site_fold.csv                        + _rowshare_raw
#    5) v0.1.1_crosstab_nurse_fold.csv                       + _rowshare_raw
#    6) v0.1.1_prevalence_device_superclass_folds1to9.csv    + _raw
#    7) v0.1.1_prevalence_site_superclass_folds1to9.csv      + _raw
#    8) v0.1.1_manifest.json
#
# ممنوعیت‌ها: آموزش مدل | AUROC/accuracy/هر متریکِ عملکردی | رسم نمودار |
#             نوشتن یا تغییر هر فایلی داخلِ پوشه‌ی دیتاست
# =====================================================================================

import os, sys, json, ast, hashlib, platform
from datetime import datetime
import numpy as np
import pandas as pd

pd.set_option("display.max_rows", 300)
pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 220)

# ------------------------------- پیکربندی -------------------------------------------
DATA_ROOT = r"D:\ptbxl-metadata-audit\data\ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.3"
OUT_DIR   = r"D:\ptbxl-metadata-audit\outputs\v0.1.1"
CODE_VERSION = "v0.1.1"
PREFIX = CODE_VERSION + "_"
SUPERCLASSES = ["NORM", "MI", "STTC", "CD", "HYP"]

# اعدادِ لنگرِ منتشرشده (برای مقایسه، نه برای اعمال) — منبع: مقاله‌ی اصلی PTB-XL
ANCHORS = {"n_records": 21799, "n_patients": 18869,
           "NORM": 9514, "MI": 5469, "STTC": 5235, "CD": 4898, "HYP": 2649}

os.makedirs(OUT_DIR, exist_ok=True)

# گاردِ ایمنی: خروجی هرگز نباید داخلِ پوشه‌ی دیتاست نوشته شود
_root_abs, _out_abs = os.path.abspath(DATA_ROOT), os.path.abspath(OUT_DIR)
assert not _out_abs.lower().startswith(_root_abs.lower()), "OUT_DIR داخلِ پوشه‌ی دیتاست است — متوقف شد."

RUN_STARTED = datetime.now().astimezone().isoformat()
OUTPUT_FILES = []          # نامِ فایل‌های تولیدشده
WARNINGS = []              # هشدارهای جمع‌شده برای خلاصه‌ی پایانی
LEAKAGE_FAIL = False


def sha256_of(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for b in iter(lambda: f.read(chunk), b""):
            h.update(b)
    return h.hexdigest()


def save_csv(obj, name, index=True):
    p = os.path.join(OUT_DIR, name)
    obj.to_csv(p, index=index, encoding="utf-8-sig")
    OUTPUT_FILES.append(name)
    print("[saved] " + p)
    return p


def hdr(t):
    print("\n" + "=" * 100)
    print(t)
    print("=" * 100)


def warn(msg):
    WARNINGS.append(msg)
    print("[WARN] " + msg)


def _json_default(o):
    if isinstance(o, np.integer):
        return int(o)
    if isinstance(o, np.floating):
        return float(o)
    if isinstance(o, np.bool_):
        return bool(o)
    if isinstance(o, np.ndarray):
        return o.tolist()
    return str(o)


# =====================================================================================
# بخش ۱ — PROVENANCE
# =====================================================================================
hdr("SECTION 1 — PROVENANCE")

print("run started         :", RUN_STARTED)
print("code version        :", CODE_VERSION)
print("python              :", sys.version.replace("\n", " "))
print("platform            :", platform.platform())
print("machine / processor :", platform.machine(), "|", platform.processor())
print("numpy               :", np.__version__)
print("pandas              :", pd.__version__)
print("dataset folder name :", os.path.basename(os.path.normpath(DATA_ROOT)))
print("dataset root        :", DATA_ROOT)
print("output dir          :", OUT_DIR)

DB_CSV      = os.path.join(DATA_ROOT, "ptbxl_database.csv")
SCP_CSV     = os.path.join(DATA_ROOT, "scp_statements.csv")
RECORDS_TXT = os.path.join(DATA_ROOT, "RECORDS")
SUMS_TXT    = os.path.join(DATA_ROOT, "SHA256SUMS")
CHANGELOGS  = ["ptbxl_v102_changelog.txt", "ptbxl_v103_changelog.txt"]

for p in (DB_CSV, SCP_CSV, RECORDS_TXT):
    assert os.path.isfile(p), "فایل پیدا نشد: " + p

print("\n--- SHA-256 فایل‌های ورودیِ اصلی (محاسبه‌شده) ---")
INPUT_HASHES = {}
for p in (DB_CSV, SCP_CSV):
    h = sha256_of(p)
    INPUT_HASHES[os.path.basename(p)] = h
    print("{:26s} {}   ({:,} bytes)".format(os.path.basename(p), h, os.path.getsize(p)))

# --- راستی‌آزمایی در برابر SHA256SUMS (فقط دو CSV؛ هیچ فایلِ سیگنالی هش نمی‌شود) ---
print("\n--- راستی‌آزمایی در برابر SHA256SUMS ---")
SHA256SUMS_VERIFICATION = {}
TARGETS = ["ptbxl_database.csv", "scp_statements.csv"]
if os.path.isfile(SUMS_TXT):
    listed = {}
    n_lines = 0
    with open(SUMS_TXT, "r", encoding="utf-8", errors="replace") as f:
        for ln in f:
            ln = ln.strip()
            if not ln:
                continue
            n_lines += 1
            parts = ln.split(None, 1)
            if len(parts) != 2:
                continue
            hval = parts[0].strip().lower()
            fname = parts[1].strip().lstrip("*").replace("\\", "/")
            base = os.path.basename(fname)
            if base in TARGETS:
                listed[base] = hval
    print("خطوطِ SHA256SUMS:", n_lines, "| سطرهای هدف پیدا شده:", len(listed))
    for t in TARGETS:
        computed = INPUT_HASHES[t].lower()
        if t not in listed:
            status, expected = "NOT_LISTED", None
        elif listed[t] == computed:
            status, expected = "MATCH", listed[t]
        else:
            status, expected = "MISMATCH", listed[t]
        SHA256SUMS_VERIFICATION[t] = {"status": status, "expected": expected, "computed": computed}
        print("  {:26s} -> {}".format(t, status))
        if status != "MATCH":
            print("      expected: {}".format(expected))
            print("      computed: {}".format(computed))
        if status == "MISMATCH":
            warn("هشِ {} با SHA256SUMS نمی‌خواند — فایل دستکاری یا ناقص است. تا حل نشود هیچ عددی معتبر نیست.".format(t))
        if status == "NOT_LISTED":
            warn("{} در SHA256SUMS فهرست نشده — یکپارچگیِ این فایل قابلِ راستی‌آزمایی نیست.".format(t))
else:
    SHA256SUMS_VERIFICATION = {"_file": "NOT_FOUND"}
    warn("SHA256SUMS در پوشه‌ی دیتاست پیدا نشد — راستی‌آزماییِ یکپارچگی انجام نشد.")

print("\n--- SHA-256 سایر ورودی‌های خوانده‌شده ---")
OTHER_HASHES = {}
for fn in [os.path.basename(RECORDS_TXT)] + CHANGELOGS:
    p = os.path.join(DATA_ROOT, fn)
    if os.path.isfile(p):
        h = sha256_of(p)
        OTHER_HASHES[fn] = h
        print("{:26s} {}   ({:,} bytes)".format(fn, h, os.path.getsize(p)))
    else:
        warn("یافت نشد: " + p)

for fn in CHANGELOGS:
    p = os.path.join(DATA_ROOT, fn)
    print("\n" + "-" * 100)
    print("--- محتوای کاملِ " + fn + " ---")
    print("-" * 100)
    if os.path.isfile(p):
        with open(p, "r", encoding="utf-8", errors="replace") as f:
            print(f.read())
    else:
        print("[WARN] فایل موجود نیست.")


# =====================================================================================
# بخش ۲ — شکلِ داده
# =====================================================================================
hdr("SECTION 2 — DATA SHAPE")

df = pd.read_csv(DB_CSV, index_col="ecg_id")

print("shape (rows, cols)      :", df.shape)
print("unique ecg_id           :", df.index.nunique())
print("duplicated ecg_id       :", int(df.index.duplicated().sum()))
print("unique patient_id       :", df["patient_id"].nunique())
print("anchor n_records/n_pat  :", ANCHORS["n_records"], "/", ANCHORS["n_patients"])
if len(df) != ANCHORS["n_records"]:
    warn("تعدادِ رکورد ({}) با عددِ لنگرِ منتشرشده ({}) یکی نیست.".format(len(df), ANCHORS["n_records"]))
if df["patient_id"].nunique() != ANCHORS["n_patients"]:
    warn("تعدادِ بیمار ({}) با عددِ لنگر ({}) یکی نیست.".format(df["patient_id"].nunique(), ANCHORS["n_patients"]))

print("\n--- ستون‌ها، dtype، missingness، cardinality ---")
inventory = pd.DataFrame({
    "column":      list(df.columns),
    "dtype":       [str(t) for t in df.dtypes],
    "n_missing":   df.isna().sum().values,
    "pct_missing": np.round(df.isna().mean().values * 100, 3),
    "n_unique":    [int(df[c].nunique(dropna=True)) for c in df.columns],
})
print(inventory.to_string(index=False))
save_csv(inventory, PREFIX + "column_inventory.csv", index=False)

# --- بررسیِ سازگاریِ RECORDS با مسیرهای داخلِ CSV (هیچ فایلِ سیگنالی باز نمی‌شود) ---
print("\n--- بررسیِ سازگاریِ RECORDS (فقط خواندنِ فهرستِ مسیرها) ---")
with open(RECORDS_TXT, "r", encoding="utf-8", errors="replace") as f:
    records_lines = [ln.strip() for ln in f if ln.strip()]
rec_set = set(records_lines)
csv_paths = set(df["filename_lr"].astype(str)) | set(df["filename_hr"].astype(str))
print("خطوطِ RECORDS                :", len(records_lines), "| یکتا:", len(rec_set))
print("مسیرهای یکتا در CSV          :", len(csv_paths))
only_csv = sorted(csv_paths - rec_set)
only_rec = sorted(rec_set - csv_paths)
print("در CSV هست ولی در RECORDS نه :", len(only_csv), ("| نمونه: " + str(only_csv[:5])) if only_csv else "")
print("در RECORDS هست ولی در CSV نه :", len(only_rec), ("| نمونه: " + str(only_rec[:5])) if only_rec else "")
if only_csv or only_rec:
    warn("عدمِ تطابقِ RECORDS با مسیرهای CSV — ممکن است صرفاً تفاوتِ قالبِ نام باشد؛ نمونه‌ها را ببین.")


# =====================================================================================
# بخش ۳ — متادیتای ضبط
# =====================================================================================
hdr("SECTION 3 — RECORDING METADATA (device / site / nurse)")

for col in ["device", "site", "nurse"]:
    print("\n--- " + col + " ---")
    print("dtype:", df[col].dtype,
          "| nunique (non-null):", int(df[col].nunique(dropna=True)),
          "| missing:", int(df[col].isna().sum()))
    vc = df[col].value_counts(dropna=False).to_frame("n")
    vc["pct"] = np.round(vc["n"] / len(df) * 100, 3)
    print(vc.to_string())

print("\n--- recording_date ---")
rd = pd.to_datetime(df["recording_date"], errors="coerce")
n_unparsable = int(rd.isna().sum() - df["recording_date"].isna().sum())
print("min:", rd.min(), "| max:", rd.max())
print("missing:", int(df["recording_date"].isna().sum()), "| unparsable:", n_unparsable)
print("\nتوزیعِ سالانه:")
print(rd.dt.year.value_counts(dropna=False).sort_index().to_frame("n").to_string())


# =====================================================================================
# بخش ۴ — دموگرافیک
# =====================================================================================
hdr("SECTION 4 — DEMOGRAPHICS")

print("--- sex ---")
vc = df["sex"].value_counts(dropna=False).to_frame("n")
vc["pct"] = np.round(vc["n"] / len(df) * 100, 3)
print(vc.to_string())

print("\n--- age ---")
age = df["age"]
print("dtype:", age.dtype, "| missing:", int(age.isna().sum()))
print("min :", age.min(), "| max:", age.max(), "| mean:", round(float(age.mean()), 3))
for q in [0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99]:
    print("  p{:g} = {}".format(q * 100, age.quantile(q)))

print("\n۱۵ مقدارِ پرتکرارِ age (برای کشفِ sentinel):")
print(age.value_counts(dropna=False).head(15).to_frame("n").to_string())

print("\nشمارشِ مقادیرِ مشکوک / سنسورشده:")
checks = [("age > 89", (age > 89)), ("age > 90", (age > 90)),
          ("age >= 100", (age >= 100)), ("age >= 120 (غیرممکنِ فیزیولوژیک)", (age >= 120)),
          ("age == 300", (age == 300)), ("age <= 0", (age <= 0))]
for name, mask in checks:
    print("  {:35s} : {}".format(name, int(mask.sum())))
print("  {:35s} : {}".format("age اعشاری (غیرصحیح)", int((age.dropna() % 1 != 0).sum())))

n_sentinel = int((age >= 120).sum())
if n_sentinel > 0:
    warn("مقدارِ sentinel در age پیدا شد ({} رکورد با age>=120). "
         "این‌ها سنین سانسورشده‌اند و باید قبل از هر مدل‌سازی صریحاً مدیریت شوند.".format(n_sentinel))

print("\nتوزیعِ کاملِ سنینِ >= ۸۰ (مرزِ سانسور معمولاً اینجاست):")
print(age[age >= 80].value_counts().sort_index().to_frame("n").to_string())

print("\n--- height / weight ---")
for c in ["height", "weight"]:
    print("\n{}: missing {} ({:.2f}%)".format(c, int(df[c].isna().sum()), df[c].isna().mean() * 100))
    print(df[c].describe().to_string())


# =====================================================================================
# بخش ۵ — سلامتِ SPLIT
# =====================================================================================
hdr("SECTION 5 — SPLIT INTEGRITY")

print("--- strat_fold ---")
fold_tab = df["strat_fold"].value_counts(dropna=False).sort_index().to_frame("n_records")
fold_tab["pct"] = np.round(fold_tab["n_records"] / len(df) * 100, 3)
fold_tab["n_patients"] = df.groupby("strat_fold")["patient_id"].nunique()
print(fold_tab.to_string())

FOLD_TAB_JSON = {}
for k, row in fold_tab.iterrows():
    key = str(int(k)) if pd.notna(k) else "NaN"
    FOLD_TAB_JSON[key] = {
        "n_records": int(row["n_records"]),
        "pct": float(row["pct"]),
        "n_patients": int(row["n_patients"]) if pd.notna(row["n_patients"]) else None,
    }

print("\n--- بررسیِ patient-disjoint بودنِ foldها ---")
folds_per_patient = df.groupby("patient_id")["strat_fold"].nunique()
violators = folds_per_patient[folds_per_patient > 1]
LEAKAGE_FAIL = len(violators) > 0
print("بیمارانی که در بیش از یک fold ظاهر می‌شوند:", len(violators))
if LEAKAGE_FAIL:
    warn("نشتِ سطح-بیمار در splitِ رسمی — این بلوکه‌کننده است.")
    print("\n۱۰ نمونه‌ی اولِ بیمارانِ متخلف و foldهایشان:")
    for pid in list(violators.index[:10]):
        sub = df[df["patient_id"] == pid]
        print("  patient_id={} | folds={} | n_records={}".format(
            pid, sorted(sub["strat_fold"].unique().tolist()), len(sub)))
else:
    print("PASS — هیچ بیماری بین دو fold مشترک نیست.")

print("\n--- توزیعِ تعدادِ رکورد به‌ازای هر بیمار ---")
rec_per_patient = df.groupby("patient_id").size()
print(rec_per_patient.value_counts().sort_index().to_frame("n_patients").to_string())
print("max records per patient:", int(rec_per_patient.max()),
      "| mean:", round(float(rec_per_patient.mean()), 4))

if "validated_by_human" in df.columns:
    print("\n--- validated_by_human × strat_fold ---")
    ct_val = pd.crosstab(df["strat_fold"], df["validated_by_human"].fillna("__MISSING__"))
    print(ct_val.to_string())

# --- بنرِ پرسروصدای نشت (تا در خروجیِ طولانی گم نشود) ---
if LEAKAGE_FAIL:
    print()
    for _ in range(5):
        print("!" * 100)
    print("!!!  LEAKAGE FAIL — نشتِ سطح-بیمار در splitِ رسمی  !!!")
    print("!!!  {} بیمار در بیش از یک strat_fold ظاهر شده‌اند.".format(len(violators)))
    print("!!!  هیچ عددِ عملکردی روی این split معتبر نیست. اجرا ادامه می‌یابد تا خروجی‌های")
    print("!!!  ممیزی تولید شوند، اما در انتهای سلول AssertionError پرتاب می‌شود.")
    for _ in range(5):
        print("!" * 100)
    print()


# =====================================================================================
# بخش ۶ — اشتقاقِ برچسب
# =====================================================================================
hdr("SECTION 6 — LABEL DERIVATION")

scp = pd.read_csv(SCP_CSV, index_col=0)
print("scp_statements shape:", scp.shape)
print("ستون‌ها:", list(scp.columns))

diag = scp[scp["diagnostic"].fillna(0) == 1]
print("\nstatements با diagnostic == 1 :", len(diag), "از", len(scp))
print("\nتوزیعِ diagnostic_class در میانِ آن‌ها:")
print(diag["diagnostic_class"].value_counts(dropna=False).to_frame("n_statements").to_string())

code2class = {k: v for k, v in diag["diagnostic_class"].items() if isinstance(v, str)}

parsed = df["scp_codes"].apply(ast.literal_eval)
all_codes = set()
for d in parsed:
    all_codes.update(d.keys())
unknown_codes = sorted(all_codes - set(scp.index))
print("\nکدهای یکتا در scp_codes :", len(all_codes),
      "| کدهایی که در scp_statements نیستند:", len(unknown_codes))
if unknown_codes:
    warn("کدهای ناشناخته در scp_codes: " + str(unknown_codes[:20]))

lik_counter = {}
for d in parsed:
    for v in d.values():
        lik_counter[v] = lik_counter.get(v, 0) + 1
print("\nتوزیعِ مقادیرِ likelihood در scp_codes (روی همه‌ی جفت‌های رکورد-کد):")
print(pd.Series(lik_counter).sort_index().to_frame("n").to_string())

extra_classes = sorted({v for v in code2class.values()} - set(SUPERCLASSES))
if extra_classes:
    warn("diagnostic_classهایی خارج از پنج ابرکلاس: " + str(extra_classes))


def to_classes(d, min_likelihood=None):
    out = set()
    for k, v in d.items():
        if k not in code2class:
            continue
        if min_likelihood is not None and not (float(v) > min_likelihood):
            continue
        out.add(code2class[k])
    return out


# قرارداد به‌کاررفته در فایلِ برچسب: همه‌ی کلیدها بدونِ فیلترِ likelihood (قراردادِ بنچمارک)
LABEL_CONVENTION = "all keys of scp_codes mapped via diagnostic==1 (no likelihood threshold)"
rec_classes = parsed.apply(lambda d: to_classes(d, None))
lab = pd.DataFrame({c: rec_classes.apply(lambda s, c=c: int(c in s)) for c in SUPERCLASSES},
                   index=df.index)

print("\n--- تعدادِ رکورد در هر ابرکلاس (قرارداد: " + LABEL_CONVENTION + ") ---")
cmp_tab = pd.DataFrame({
    "derived": [int(lab[c].sum()) for c in SUPERCLASSES],
    "anchor_published": [ANCHORS[c] for c in SUPERCLASSES],
})
cmp_tab.index = SUPERCLASSES
cmp_tab["diff"] = cmp_tab["derived"] - cmp_tab["anchor_published"]
cmp_tab["prevalence_pct"] = np.round(cmp_tab["derived"] / len(df) * 100, 3)
print(cmp_tab.to_string())
if (cmp_tab["diff"] != 0).any():
    warn("اعدادِ مشتق‌شده با اعدادِ منتشرشده اختلاف دارند — علتِ محتمل: تغییرِ نسخه‌ی دیتاست "
         "یا تفاوتِ قرارداد. changelogهای بخش ۱ را در گزارش ذکر کن.")

n_labels = lab.sum(axis=1)
print("\nرکوردهای بدونِ هیچ ابرکلاس :", int((n_labels == 0).sum()),
      "({:.2f}%)".format((n_labels == 0).mean() * 100))
print("\nتوزیعِ تعدادِ برچسب به‌ازای هر رکورد:")
nl = n_labels.value_counts().sort_index().to_frame("n_records")
nl["pct"] = np.round(nl["n_records"] / len(df) * 100, 3)
print(nl.to_string())

alt = parsed.apply(lambda d: to_classes(d, 0.0))
n_changed = int((rec_classes != alt).sum())
print("\nحساسیت: اگر فقط کدهای با likelihood>0 نگه داشته شوند،",
      n_changed, "رکورد ({:.2f}%) مجموعه‌ی برچسبش تغییر می‌کند.".format(n_changed / len(df) * 100))

labels_out = pd.concat([df[["patient_id", "strat_fold"]], lab], axis=1).reset_index()
save_csv(labels_out, PREFIX + "labels.csv", index=False)


# =====================================================================================
# بخش ۷ — جدول‌های تقاطعی  (مهم‌ترین بخش)
# =====================================================================================
hdr("SECTION 7 — CROSSTABS (device / site / nurse  ×  fold و ابرکلاس)")


def cramers_v(ct):
    """Cramér's V خام و تصحیح‌شده‌ی Bergsma، بدونِ وابستگی به scipy."""
    obs = ct.values.astype(float)
    n = obs.sum()
    row = obs.sum(axis=1, keepdims=True)
    col = obs.sum(axis=0, keepdims=True)
    exp = row @ col / n
    with np.errstate(divide="ignore", invalid="ignore"):
        chi2 = float(np.nansum(np.where(exp > 0, (obs - exp) ** 2 / np.where(exp > 0, exp, 1.0), 0.0)))
    r, k = obs.shape
    dof = (r - 1) * (k - 1)
    v = float(np.sqrt(chi2 / (n * max(min(r, k) - 1, 1))))
    phi2 = chi2 / n
    phi2c = max(0.0, phi2 - dof / (n - 1))
    rc = r - (r - 1) ** 2 / (n - 1)
    kc = k - (k - 1) ** 2 / (n - 1)
    denom = max(min(rc, kc) - 1, 1e-12)
    vc = float(np.sqrt(max(phi2c, 0.0) / denom))
    return {"n": int(n), "shape": [r, k], "chi2": chi2, "dof": dof,
            "cramers_v": v, "cramers_v_biascorrected": vc}


def to_sortable_label(s, name, missing="__MISSING__"):
    """برچسبِ رشته‌ایِ صفرپرشده برای ستون‌های عددی (مثلاً site_007) تا مرتب‌سازی لغوی درست باشد."""
    if pd.api.types.is_numeric_dtype(s):
        nn = s.dropna()
        integral = bool(len(nn) == 0 or np.all(np.isclose(nn.values, np.round(nn.values))))
        if integral:
            width = max(3, len(str(int(nn.max()))) if len(nn) else 3)
            out = s.apply(lambda v: missing if pd.isna(v) else "{}_{}".format(name, str(int(v)).zfill(width)))
            return out.astype(str), True
    return s.fillna(missing).astype(str).str.strip(), False


dfx = df.copy()
print("--- ساختِ برچسب‌های مرتب‌شدنی ---")
for c in ["device", "site", "nurse"]:
    dfx[c], padded = to_sortable_label(df[c], c)
    print("  {:7s} zero-padded: {:5s} | نمونه: {}".format(
        c, str(padded), sorted(dfx[c].unique())[:3]))

mask_1to9 = dfx["strat_fold"] <= 9
mask_f10 = dfx["strat_fold"] == 10
print("\nرکوردها: folds 1–9 = {} | fold 10 = {} | مجموع = {}".format(
    int(mask_1to9.sum()), int(mask_f10.sum()), len(dfx)))

CRAMER = {}

# --- ۷.۱–۷.۳: device / site / nurse × strat_fold (روی کلِ دیتاست) ---
for col, key in [("device", "device_x_fold"), ("site", "site_x_fold"), ("nurse", "nurse_x_fold")]:
    fname = PREFIX + "crosstab_" + col + "_fold.csv"
    print("\n" + "-" * 100)
    print("--- " + col + " × strat_fold (شمارش، کلِ دیتاست) ---")
    ct = pd.crosstab(dfx[col], dfx["strat_fold"]).sort_index()
    ct_full = ct.copy()
    ct_full["TOTAL"] = ct.sum(axis=1)
    print(ct_full.to_string())
    rowshare = ct.div(ct.sum(axis=1), axis=0)
    print("\nسهمِ سطری (٪ هر " + col + " در هر fold):")
    print((rowshare * 100).round(2).to_string())
    stats = cramers_v(ct)
    CRAMER[key] = stats
    print("\nCramér's V ({} × strat_fold): {:.4f}  | bias-corrected: {:.4f}  "
          "| chi2={:.2f}, dof={}, n={}, shape={}".format(
              col, stats["cramers_v"], stats["cramers_v_biascorrected"],
              stats["chi2"], stats["dof"], stats["n"], stats["shape"]))
    save_csv(ct_full, fname, index=True)
    save_csv(rowshare, fname.replace(".csv", "_rowshare_raw.csv"), index=True)

# --- ۷.۴: شیوعِ ابرکلاس‌ها درونِ هر device / site — فقط folds 1–9 ---
# فولدِ ۱۰ (تست) عمداً کور نگه داشته می‌شود: هیچ جدولِ مشترکِ متادیتا × برچسب روی آن ساخته نمی‌شود.
lab_1to9 = lab.loc[mask_1to9]
overall_prev_1to9 = lab_1to9.mean()
for col in ["device", "site"]:
    fname = PREFIX + "prevalence_" + col + "_superclass_folds1to9.csv"
    print("\n" + "-" * 100)
    print("--- نرخِ شیوعِ ابرکلاس‌ها درونِ هر " + col + "  [فقط folds 1–9؛ فولدِ ۱۰ کور] ---")
    joined = dfx.loc[mask_1to9, [col]].join(lab_1to9)
    prev = joined.groupby(col)[SUPERCLASSES].mean()
    prev.insert(0, "n_records", joined.groupby(col).size())
    prev = prev.sort_values("n_records", ascending=False)

    disp = prev.copy()
    for c in SUPERCLASSES:
        disp[c] = (disp[c] * 100).round(2)
    overall_row = pd.DataFrame(
        [[int(mask_1to9.sum())] + [round(float(overall_prev_1to9[c]) * 100, 2) for c in SUPERCLASSES]],
        index=["__OVERALL_folds1to9__"], columns=["n_records"] + SUPERCLASSES)
    disp = pd.concat([overall_row, disp])
    print("(اعداد بر حسب درصد؛ سطرِ __OVERALL_folds1to9__ خطِ مبنای همان زیرمجموعه است)")
    print(disp.to_string())

    print("\nبیشینه‌ی فاصله از شیوعِ کلیِ folds 1–9، بر حسب واحدِ درصد:")
    spread = (prev[SUPERCLASSES].sub(overall_prev_1to9, axis=1).abs() * 100)
    print(spread.max().round(2).to_frame("max_abs_deviation_pp").to_string())

    save_csv(disp, fname, index=True)
    save_csv(prev, fname.replace(".csv", "_raw.csv"), index=True)

# --- شیوعِ حاشیه‌ایِ فولدِ ۱۰ = خطِ پایه‌ی شانسِ AUPRC (فقط حاشیه‌ای، بدونِ هیچ تقاطعی) ---
print("\n" + "-" * 100)
print("--- خطِ پایه‌ی شانسِ AUPRC روی فولدِ ۱۰ (فقط شیوعِ حاشیه‌ای) ---")
lab_f10 = lab.loc[mask_f10]
CHANCE_F10 = {"n_records_fold10": int(mask_f10.sum()), "prevalence": {}, "counts": {}}
rows = []
for c in SUPERCLASSES:
    cnt = int(lab_f10[c].sum())
    prv = float(lab_f10[c].mean()) if len(lab_f10) else float("nan")
    CHANCE_F10["counts"][c] = cnt
    CHANCE_F10["prevalence"][c] = prv
    rows.append({"superclass": c, "n_positive": cnt, "prevalence": round(prv, 6),
                 "prevalence_pct": round(prv * 100, 3)})
f10_tab = pd.DataFrame(rows).set_index("superclass")
print("n_records در فولدِ ۱۰:", int(mask_f10.sum()))
print(f10_tab.to_string())
print("\nیادآوری: AUPRCِ شانس برای هر کلاس = همین شیوع. macro-AUROCِ شانس = 0.50.")
print("هیچ جدولِ device/site × برچسبی روی فولدِ ۱۰ محاسبه نشد (کوریِ عمدیِ تست).")


# --- ۷.۵: دسته‌های دیده‌نشده‌ی متادیتا بین foldها ---
print("\n" + "-" * 100)
print("--- ۷.۵ — دسته‌های دیده‌نشده‌ی device / site / nurse بین foldها ---")
UNSEEN = {}
for col in ["device", "site", "nurse"]:
    sub = dfx[[col, "strat_fold"]]
    cats_train = set(sub.loc[sub["strat_fold"] <= 8, col].unique())
    cats_f9 = set(sub.loc[sub["strat_fold"] == 9, col].unique())
    cats_f10 = set(sub.loc[sub["strat_fold"] == 10, col].unique())

    n_in_f9 = sub[sub["strat_fold"] == 9][col].value_counts()
    n_in_f10 = sub[sub["strat_fold"] == 10][col].value_counts()
    n_overall = sub[col].value_counts()

    new9 = sorted(cats_f9 - cats_train)
    new10 = sorted(cats_f10 - cats_train)
    nfolds = sub.groupby(col)["strat_fold"].nunique()
    single = sorted(nfolds[nfolds == 1].index)

    UNSEEN[col] = {
        "n_categories_total": int(sub[col].nunique()),
        "n_categories_in_folds1to8": len(cats_train),
        "in_fold9_not_in_folds1to8": [
            {"category": c, "n_records_in_fold9": int(n_in_f9.get(c, 0))} for c in new9],
        "in_fold10_not_in_folds1to8": [
            {"category": c, "n_records_in_fold10": int(n_in_f10.get(c, 0))} for c in new10],
        "single_fold_only": [
            {"category": c,
             "fold": int(sub.loc[sub[col] == c, "strat_fold"].iloc[0]),
             "n_records": int(n_overall.get(c, 0))} for c in single],
    }

    print("\n--- " + col + " ---")
    print("دسته‌های یکتا: {} | در folds 1–8: {}".format(
        UNSEEN[col]["n_categories_total"], UNSEEN[col]["n_categories_in_folds1to8"]))
    print("در فولدِ ۹ ولی نه در ۱–۸ ({}): {}".format(
        len(new9), UNSEEN[col]["in_fold9_not_in_folds1to8"] if new9 else "هیچ"))
    print("در فولدِ ۱۰ ولی نه در ۱–۸ ({}): {}".format(
        len(new10), UNSEEN[col]["in_fold10_not_in_folds1to8"] if new10 else "هیچ"))
    print("فقط در یک fold ({}): {}".format(
        len(single), UNSEEN[col]["single_fold_only"] if single else "هیچ"))
    if new9 or new10:
        warn("{}: دسته‌های دیده‌نشده در val/test وجود دارد — encoding باید دسته‌ی ناشناخته را "
             "صریحاً مدیریت کند وگرنه هنگام inference می‌شکند یا بی‌سروصدا صفر می‌شود.".format(col))


# =====================================================================================
# بخش ۸ — MANIFEST
# =====================================================================================
hdr("SECTION 8 — MANIFEST")

manifest = {
    "code_version": CODE_VERSION,
    "run_started": RUN_STARTED,
    "run_finished": datetime.now().astimezone().isoformat(),
    "dataset_root": DATA_ROOT,
    "dataset_folder_name": os.path.basename(os.path.normpath(DATA_ROOT)),
    "output_dir": OUT_DIR,
    "input_files_sha256": INPUT_HASHES,
    "other_inputs_sha256": OTHER_HASHES,
    "sha256sums_verification": SHA256SUMS_VERIFICATION,
    "library_versions": {
        "python": sys.version.replace("\n", " "),
        "numpy": np.__version__,
        "pandas": pd.__version__,
        "platform": platform.platform(),
    },
    "dataset_summary": {
        "n_records": int(len(df)),
        "n_patients": int(df["patient_id"].nunique()),
        "n_devices": int(df["device"].nunique(dropna=True)),
        "n_sites": int(df["site"].nunique(dropna=True)),
        "n_nurses": int(df["nurse"].nunique(dropna=True)),
        "superclass_counts": {c: int(lab[c].sum()) for c in SUPERCLASSES},
        "n_records_without_superclass": int((n_labels == 0).sum()),
        "n_age_sentinel_ge120": n_sentinel,
    },
    "fold_tab": FOLD_TAB_JSON,
    "label_convention": LABEL_CONVENTION,
    "label_likelihood_sensitivity": {
        "alt_rule": "keep only codes with likelihood > 0",
        "n_changed": n_changed,
        "pct_changed": round(n_changed / len(df) * 100, 4),
    },
    "cramers_v": CRAMER,
    "auprc_chance_baseline_fold10": CHANCE_F10,
    "unseen_categories": UNSEEN,
    "checks": {
        "patient_disjoint_folds": "FAIL" if LEAKAGE_FAIL else "PASS",
        "n_patients_in_multiple_folds": int(len(violators)),
        "records_manifest_mismatch_csv_only": len(only_csv),
        "records_manifest_mismatch_records_only": len(only_rec),
    },
    "warnings": WARNINGS,
    "outputs": [],
}

for name in OUTPUT_FILES:
    p = os.path.join(OUT_DIR, name)
    manifest["outputs"].append({"file": name, "bytes": os.path.getsize(p), "sha256": sha256_of(p)})

MANIFEST_PATH = os.path.join(OUT_DIR, PREFIX + "manifest.json")
with open(MANIFEST_PATH, "w", encoding="utf-8") as f:
    json.dump(manifest, f, ensure_ascii=False, indent=2, default=_json_default)
print("[saved] " + MANIFEST_PATH)
print("SHA-256 " + PREFIX + "manifest.json:", sha256_of(MANIFEST_PATH))

print("\n--- محتوای manifest ---")
print(json.dumps(manifest, ensure_ascii=False, indent=2, default=_json_default))


# =====================================================================================
# خلاصه‌ی پایانی (برای گزارشِ مرحله)
# =====================================================================================
hdr("SUMMARY — " + CODE_VERSION)
print("records / patients        :", len(df), "/", df["patient_id"].nunique())
print("device / site / nurse     :", df["device"].nunique(dropna=True), "/",
      df["site"].nunique(dropna=True), "/", df["nurse"].nunique(dropna=True))
print("SHA256SUMS verification   :", {k: (v["status"] if isinstance(v, dict) and "status" in v else v)
                                      for k, v in SHA256SUMS_VERIFICATION.items()})
print("age sentinel (>=120)      :", n_sentinel)
print("بدونِ ابرکلاس              :", int((n_labels == 0).sum()))
print("چندبرچسبی (>=2)           :", int((n_labels >= 2).sum()))
print("patient-disjoint folds    :", "FAIL" if LEAKAGE_FAIL else "PASS")
for key, colname in [("device_x_fold", "device"), ("site_x_fold", "site"), ("nurse_x_fold", "nurse")]:
    print("Cramér's V {:6s}×fold    : {:.4f} (corrected {:.4f})".format(
        colname, CRAMER[key]["cramers_v"], CRAMER[key]["cramers_v_biascorrected"]))
print("AUPRC chance (fold 10)    :",
      {c: round(CHANCE_F10["prevalence"][c], 4) for c in SUPERCLASSES})
print("دسته‌های دیده‌نشده در val/test:",
      {c: len(UNSEEN[c]["in_fold9_not_in_folds1to8"]) + len(UNSEEN[c]["in_fold10_not_in_folds1to8"])
       for c in ["device", "site", "nurse"]})
print("فایل‌های تولیدشده          :", len(OUTPUT_FILES) + 1)
print("\nهشدارها ({}):".format(len(WARNINGS)))
for w in WARNINGS:
    print("  - " + w)

if LEAKAGE_FAIL:
    raise AssertionError(
        "نشتِ سطح-بیمار: {} بیمار در بیش از یک strat_fold ظاهر شده‌اند. "
        "خروجی‌های ممیزی نوشته شدند؛ قبل از هر مدل‌سازی این را حل کن.".format(len(violators)))

print("\n" + CODE_VERSION + " با موفقیت تمام شد.")


SECTION 1 — PROVENANCE
run started         : 2026-09-10T01:26:26.446353+03:30
code version        : v0.1.1
python              : 3.10.20 | packaged by Anaconda, Inc. | (main, Jun 11 2026, 15:13:20) [MSC v.1942 64 bit (AMD64)]
platform            : Windows-10-10.0.19045-SP0
machine / processor : AMD64 | Intel64 Family 6 Model 186 Stepping 2, GenuineIntel
numpy               : 1.26.4
pandas              : 2.3.3
dataset folder name : ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.3
dataset root        : D:\ptbxl-metadata-audit\data\ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.3
output dir          : D:\ptbxl-metadata-audit\outputs\v0.1.1

--- SHA-256 فایل‌های ورودیِ اصلی (محاسبه‌شده) ---
ptbxl_database.csv         7600de9c1b27d181d850b3c6038a35d7c3ddb6bb33b702e3a20252a6859d216b   (6,594,879 bytes)
scp_statements.csv         ad05b0b1fcae83bb1230755ad9cfc7c96f303feddc08a4a9ad5bdc9ca63bac8f   (9,720 bytes)

--- راستی‌آزمایی در برابر SHA256SUMS ---
[WA